# Femmestral — Mistral 7B Fine-Tune

Fine-tunes Mistral 7B on women's health misinformation using Unsloth + LoRA.  
Logs training to W&B Models, traces inference with Weave, pushes adapter to HuggingFace.

**Before running:** Add these to Colab Secrets (left sidebar → key icon):
- `WANDB_API_KEY`
- `HF_TOKEN`
- `GITHUB_TOKEN` (classic PAT with repo read access)

In [ ]:
# Cell 1 — Install dependencies
%%capture
!pip install unsloth
!pip install trl transformers accelerate peft bitsandbytes
!pip install wandb weave huggingface_hub datasets

In [ ]:
# Cell 2 — Authenticate
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"]  = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"]       = userdata.get("HF_TOKEN")
os.environ["WANDB_PROJECT"]  = "femmestral"

import wandb
import weave

run = wandb.init(
    project="femmestral",
    job_type="fine-tune",
    config={
        "base_model":      "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
        "lora_r":          16,
        "lora_alpha":      16,
        "training_steps":  100,
        "learning_rate":   1e-4,
        "batch_size":      2,
        "grad_accum":      4,
        "max_seq_length":  2048,
        "train_examples":  240,
        "eval_examples":   60,
        "task":            "women_health_misinformation",
    }
)
weave.init("femmestral")
print("W&B and Weave initialised.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: elenaajayi (elenaajayi-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Initializing weave.
weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


W&B and Weave initialised.


In [ ]:
  # Cell 3 — Clone repo and load data
  import subprocess, json

  # Remove existing directory if it exists from a previous attempt
  subprocess.run(["rm", "-rf", "femmestral"], check=False)

  subprocess.run(
      ["git", "clone", "--branch", "elena_mistral_hack",
       "https://github.com/elenaajayi/femmestral.git"],
      check=True
  )

  def load_jsonl(path):
      with open(path) as f:
          return [json.loads(line) for line in f if line.strip()]

  train_data = load_jsonl("femmestral/data/train.jsonl")
  eval_data  = load_jsonl("femmestral/data/eval.jsonl")
  print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")



Train: 240 | Eval: 60


In [ ]:
# Cell 4 — Load Mistral 7B in 4-bit via Unsloth
from unsloth import FastLanguageModel
import torch

max_seq_length = run.config["max_seq_length"]

model, tokenizer = FastLanguageModel.from_pretrained(
      model_name    = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit" ,
      max_seq_length= max_seq_length,
      dtype         = None,
      load_in_4bit  = True,
)

print("Model loaded.")

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Model loaded.


In [ ]:
# Cell 5 — Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r                      = run.config["lora_r"],
    target_modules         = ["q_proj", "k_proj", "v_proj", "o_proj",
                               "gate_proj", "up_proj", "down_proj"],
    lora_alpha             = run.config["lora_alpha"],
    lora_dropout           = 0,
    bias                   = "none",
    use_gradient_checkpointing = "unsloth",
    random_state           = 42,
)
print("LoRA adapters applied.")

Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA adapters applied.


In [ ]:
# Cell 6 — Prepare datasets
from datasets import Dataset

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data).map(format_example)
eval_dataset  = Dataset.from_list(eval_data).map(format_example)
print(f"Datasets formatted. Example:\n{train_dataset[0]['text'][:300]}")

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Datasets formatted. Example:
<s>[INST] Deodorant with aluminium causes breast cancer — chemicals go directly from armpit to breast tissue!! Switch to natural deodorant NOW!![/INST] **Verdict: False**
**Confidence: High** | **Evidence: B** | **Severity: High**

The National Cancer Institute and Cancer Research UK have concluded 


In [ ]:
# Cell 7 — Train
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_dataset,
    eval_dataset      = eval_dataset,
    dataset_text_field= "text",
    max_seq_length    = max_seq_length,
    dataset_num_proc  = 2,
    args = TrainingArguments(
        per_device_train_batch_size = run.config["batch_size"],
        gradient_accumulation_steps = run.config["grad_accum"],
        warmup_steps                = 5,
        max_steps                   = run.config["training_steps"],
        learning_rate               = run.config["learning_rate"],
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        output_dir                  = "outputs",
        report_to                   = "wandb",
        eval_strategy               = "steps",
        eval_steps                  = 20,
        save_strategy               = "steps",
        save_steps                  = 20,
        load_best_model_at_end      = True,
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/240 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 240 | Num Epochs = 4 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss,Validation Loss
20,1.116500,1.126291
40,0.845100,0.992846
60,0.806100,0.925821
80,0.588900,0.964149
100,0.535800,0.940247


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▃▁▂▂
eval/runtime,█▂▂▁▁
eval/samples_per_second,▁▇▇██
eval/steps_per_second,▁▇▇██
train/epoch,▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,█▅▃ ▄▁▁▁▁▁▁▁▁▁▂▁▁▂▂▂▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂
train/learning_rate,▄▇███████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▂▂▂▂▁▁▁▁▁
train/loss,██▅▅▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▁▁▁▁▁▁▁▁▁
eval/loss,0.94025
eval/runtime,14.326


TrainOutput(global_step=100, training_loss=0.9204305186867714, metrics={'train_runtime': 505.8632, 'train_samples_per_second': 1.581, 'train_steps_per_second': 0.198, 'total_flos': 6872696061935616.0, 'train_loss': 0.9204305186867714, 'epoch': 3.3333333333333335})

In [ ]:
# Cell 8 — Push LoRA adapter to HuggingFace
HF_REPO = "elenaajayi/femmestral-mistral-7b"

model.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
print(f"Pushed to https://huggingface.co/{HF_REPO}")

Saved model to https://huggingface.co/elenaajayi/femmestral-mistral-7b
Pushed to https://huggingface.co/elenaajayi/femmestral-mistral-7b


In [ ]:
  # Cell 9 — Log W&B artifact
  run = wandb.init(project="femmestral", resume="must", id="ogejyi0g")

  artifact = wandb.Artifact(
      name        = "mistral-7b-femmestral",
      type        = "model",
      description = "Mistral 7B LoRA adapter fine-tuned on 240 women's health misinformation examples.",
      metadata    = {
          "base_model":      "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
          "lora_r":          16,
          "training_steps":  100,
          "learning_rate":   1e-4,
          "train_examples":  240,
          "eval_examples":   60,
          "task":            "women_health_misinformation",
          "hf_repo":         HF_REPO,
          "access":          f"PeftModel.from_pretrained(base, '{HF_REPO}')",
      }
  )

  artifact.add_file("femmestral/data/train.jsonl", name="train.jsonl")
  artifact.add_file("femmestral/data/eval.jsonl",  name="eval.jsonl")

  with open("/tmp/model_ref.txt", "w") as f:
      f.write(f"LoRA adapter: https://huggingface.co/{HF_REPO}\n\n")
      f.write("Load:\n")
      f.write("  from peft import PeftModel\n")
      f.write("  from transformers import AutoModelForCausalLM\n")
      f.write(f"  base = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-Instruct-v0.3')\n")
      f.write(f"  model = PeftModel.from_pretrained(base, '{HF_REPO}')\n")

  artifact.add_file("/tmp/model_ref.txt", name="model_ref.txt")
  run.log_artifact(artifact)
  print("W&B artifact logged.")

wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


W&B artifact logged.


In [ ]:
# Cell 10 — Sample inference with Weave tracing
FastLanguageModel.for_inference(model)

@weave.op()
def classify_claim(claim: str) -> str:
    messages = [{"role": "user", "content": claim}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids      = inputs,
        max_new_tokens = 600,
        temperature    = 0.3,
        do_sample      = True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

test_claims = [
    "HPV vaccine causes infertility!! Share with all women!!!",
    "Eating papaya causes miscarriage. Warn all pregnant women!!",
    "Birth control pills cause permanent infertility after 2 years of use.",
    "Mammograms cause breast cancer from the radiation.",
    "Women cannot get pregnant while breastfeeding — it is 100% safe.",
]

results = []
for claim in test_claims:
    response = classify_claim(claim)
    results.append({"claim": claim, "response": response})
    print(f"\nClaim: {claim}\nResponse: {response[:300]}...\n{'='*60}")

run.log({"sample_outputs": wandb.Table(
    columns=["claim", "response"],
    data=[[r["claim"], r["response"]] for r in results]
)})

wandb.finish()
print("Done.")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-2864-7c65-ab47-94e5edd10e1f
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-53a5-7dba-b81f-4c2abe02226c



Claim: HPV vaccine causes infertility!! Share with all women!!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Multiple large studies have found no evidence that the HPV vaccine affects fertility. The vaccine protects against cervical cancer and other HPV-related cancers. It is safe and recommended for adolescents.

**Source:** C...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-7ad1-7b90-9c34-635ade040ab0



Claim: Eating papaya causes miscarriage. Warn all pregnant women!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Papaya is safe during pregnancy. The myth that papaya causes miscarriage has no scientific basis. Green, unripe papaya contains latex that may stimulate contractions in large amounts, but ripe papaya is safe.

**Source...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-9b5a-7b69-aeb5-75be304690b3



Claim: Birth control pills cause permanent infertility after 2 years of use.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Birth control pills do not cause permanent infertility. Fertility usually returns within 3–6 months of stopping. The pill affects ovulation temporarily while being taken.

**Source:** NIH Office on Women's Health — Con...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca695-c46f-71fd-8c89-35e103e44b92



Claim: Mammograms cause breast cancer from the radiation.
Response: **Verdict: False**
**Confidence: High** | **Evidence: B** | **Severity: High**

Mammograms use very low doses of radiation. The risk from a single mammogram is extremely small. The benefit of early cancer detection far outweighs this minuscule risk.

**Source:** NIH — Mammography and Radiation Risk
...

Claim: Women cannot get pregnant while breastfeeding — it is 100% safe.
Response: **Verdict: Misleading**
**Confidence: Medium** | **Evidence: A** | **Severity: Medium**

The Lactational Amenorrhoea Method (LAM) is a form of natural family planning that can work for some women, but it is not 100% effective and is not a reliable method of contraception. Breastfeeding does delay ov...


eval/loss,0.94025
eval/runtime,14.326
eval/samples_per_second,4.188
eval/steps_per_second,2.094
total_flos,6872696061935616
train/epoch,3.33333
train/global_step,100
train/grad_norm,1.6414
train/learning_rate,0.0
train/loss,0.5358
+4,...


Done.


In [ ]:
# %%capture
!pip install wandb weave

import os
import wandb
import weave
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"]      = userdata.get("HF_TOKEN")

run2 = wandb.init(
    project="femmestral",
    job_type="fine-tune",
    config={
        "base_model":      "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
        "lora_r":          16,
        "lora_alpha":      16,
        "training_steps":  80,
        "learning_rate":   5e-5,
        "batch_size":      2,
        "grad_accum":      4,
        "max_seq_length":  2048,
        "train_examples":  240,
        "eval_examples":   60,
        "task":            "women_health_misinformation",
        "note":            "Run 2 — lower LR to reduce overfitting",
    }
)
weave.init("femmestral")
print("W&B run initialised.")

wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


W&B run initialised.


In [ ]:

  # Run 2 — Cell B: Install + Load model
  # %%capture
  !pip install unsloth trl transformers accelerate peft bitsandbytes datasets

  from unsloth import FastLanguageModel
  from unsloth import is_bfloat16_supported
  import torch

  model2, tokenizer2 = FastLanguageModel.from_pretrained(
      model_name    = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
      max_seq_length= 2048,
      dtype         = None,
      load_in_4bit  = True,
  )

  model2 = FastLanguageModel.get_peft_model(
      model2,
      r                          = 16,
      target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                     "gate_proj", "up_proj", "down_proj"],
      lora_alpha                 = 16,
      lora_dropout               = 0,
      bias                       = "none",
      use_gradient_checkpointing = "unsloth",
      random_state               = 42,
  )
  print("Model 2 loaded and LoRA applied.")

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model 2 loaded and LoRA applied.


In [ ]:
  # Run 2 — Cell C: Load data + Train
  import subprocess, json
  from datasets import Dataset
  from trl import SFTTrainer
  from transformers import TrainingArguments

  # Reload data (in case of disconnect)
  subprocess.run(["rm", "-rf", "femmestral"], check=False)
  subprocess.run(["git", "clone", "--branch", "elena_mistral_hack",
                  "https://github.com/elenaajayi/femmestral.git"], check=True)

  def load_jsonl(path):
      with open(path) as f:
          return [json.loads(line) for line in f if line.strip()]

  train_data = load_jsonl("femmestral/data/train.jsonl")
  eval_data  = load_jsonl("femmestral/data/eval.jsonl")

  def format_example(example):
      text = tokenizer2.apply_chat_template(
          example["messages"], tokenize=False, add_generation_prompt=False
      )
      return {"text": text}

  train_dataset = Dataset.from_list(train_data).map(format_example)
  eval_dataset  = Dataset.from_list(eval_data).map(format_example)

  trainer2 = SFTTrainer(
      model             = model2,
      tokenizer         = tokenizer2,
      train_dataset     = train_dataset,
      eval_dataset      = eval_dataset,
      dataset_text_field= "text",
      max_seq_length    = 2048,
      dataset_num_proc  = 2,
      args = TrainingArguments(
          per_device_train_batch_size = 2,
          gradient_accumulation_steps = 4,
          warmup_steps                = 5,
          max_steps                   = 80,
          learning_rate               = 5e-5,
          fp16                        = not is_bfloat16_supported(),
          bf16                        = is_bfloat16_supported(),
          logging_steps               = 1,
          optim                       = "adamw_8bit",
          weight_decay                = 0.01,
          lr_scheduler_type           = "cosine",
          seed                        = 42,
          output_dir                  = "outputs2",
          report_to                   = "wandb",
          eval_strategy               = "steps",
          eval_steps                  = 20,
          save_strategy               = "steps",
          save_steps                  = 20,
          load_best_model_at_end      = True,
      ),
  )
  trainer2.train()

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/240 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 240 | Num Epochs = 3 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)


Step,Training Loss,Validation Loss
20,1.332200,1.315043
40,1.022900,1.082266
60,1.001100,1.033853
80,0.891900,1.026828


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


eval/loss,█▂▁▁
eval/runtime,█▁▁▁
eval/samples_per_second,▁▇██
eval/steps_per_second,▁▇██
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/grad_norm,█▄▄▂ ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▁▂▁▂▂▁▁▁▁
train/learning_rate,▁▂▄▇██████▇▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,███▅▄▄▃▃▃▃▂▃▂▃▂▂▂▂▂▂▂▂▁▂▁▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁
eval/loss,1.02683
eval/runtime,14.2932


TrainOutput(global_step=80, training_loss=1.23151036798954, metrics={'train_runtime': 445.2653, 'train_samples_per_second': 1.437, 'train_steps_per_second': 0.18, 'total_flos': 5494842306674688.0, 'train_loss': 1.23151036798954, 'epoch': 2.6666666666666665})

In [16]:
  # Run 2 — Cell D: Push to HuggingFace + log artifact
  import wandb
  import os
  from google.colab import userdata

  os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

  HF_REPO2 = "elenaajayi/femmestral-mistral-7b-v2"

  # Resume run 2
  run2 = wandb.init(project="femmestral", resume="must", id="971dkrfv")

  model2.push_to_hub(HF_REPO2, token=os.environ["HF_TOKEN"])
  tokenizer2.push_to_hub(HF_REPO2, token=os.environ["HF_TOKEN"])
  print(f"Pushed to https://huggingface.co/{HF_REPO2}")

  artifact2 = wandb.Artifact(
      name        = "mistral-7b-femmestral-v2",
      type        = "model",
      description = "Run 2 — lower LR (5e-5), 80 steps, reduced overfitting.",
      metadata    = {
          "base_model":     "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
          "learning_rate":  5e-5,
          "training_steps": 80,
          "hf_repo":        HF_REPO2,
      }
  )
  artifact2.add_file("femmestral/data/train.jsonl", name="train.jsonl")
  artifact2.add_file("femmestral/data/eval.jsonl",  name="eval.jsonl")
  run2.log_artifact(artifact2)
  print("Artifact logged.")

wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: elenaajayi.
weave: View Weave data at https://wandb.ai/elenaajayi-n-a/femmestral/weave


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pvp5chue7/tokenizer.model: 100%|##########|  587kB /  587kB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/elenaajayi/femmestral-mistral-7b-v2
Artifact logged.


In [17]:

  # Run 2 — Cell E: Inference + Weave tracing + finish
  import wandb
  import weave
  from unsloth import FastLanguageModel

  FastLanguageModel.for_inference(model2)

  @weave.op()
  def classify_claim_v2(claim: str) -> str:
      messages = [{"role": "user", "content": claim}]
      inputs = tokenizer2.apply_chat_template(
          messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
      ).to("cuda")
      attention_mask = inputs.ne(tokenizer2.pad_token_id)
      outputs = model2.generate(
          input_ids      = inputs,
          attention_mask = attention_mask,
          max_new_tokens = 600,
          temperature    = 0.3,
          do_sample      = True,
      )
      return tokenizer2.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

  test_claims = [
      "HPV vaccine causes infertility!! Share with all women!!!",
      "Eating papaya causes miscarriage. Warn all pregnant women!!",
      "Birth control pills cause permanent infertility after 2 years of use.",
      "Mammograms cause breast cancer from the radiation.",
      "Women cannot get pregnant while breastfeeding — it is 100% safe.",
  ]

  results2 = []
  for claim in test_claims:
      response = classify_claim_v2(claim)
      results2.append({"claim": claim, "response": response})
      print(f"\nClaim: {claim}\nResponse: {response[:300]}...\n{'='*60}")

  run2.log({"sample_outputs": wandb.Table(
      columns=["claim", "response"],
      data=[[r["claim"], r["response"]] for r in results2]
  )})

  wandb.finish()
  print("Run 2 done.")

weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-2549-7bbf-a34a-ae9fa1a76969
weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-57a9-72e7-91b9-7a5890132ef5



Claim: HPV vaccine causes infertility!! Share with all women!!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Multiple large studies have found no evidence that the HPV vaccine affects fertility. The vaccine protects against cervical cancer and other HPV-related cancers and diseases. It does not cause infertility.

**Source:** W...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-8253-7d5c-9a97-be4e7f6bf75c



Claim: Eating papaya causes miscarriage. Warn all pregnant women!!
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Papaya is safe during pregnancy. The unripe fruit contains latex that can stimulate uterine contractions in large amounts, but ripe papaya is safe and rich in vitamins A and C. This claim is not supported by evidence.

*...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-a96b-7523-821a-dd3f17ae47cc



Claim: Birth control pills cause permanent infertility after 2 years of use.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Birth control pills do not cause permanent infertility. Fertility returns within 3–6 months of stopping. The pill does not affect the number or quality of eggs.

**Source:** WHO Medical Eligibility Criteria for Contracep...


weave: 🍩 https://wandb.ai/elenaajayi-n-a/femmestral/r/call/019ca6c6-d327-7491-bb26-08ab8f00b762



Claim: Mammograms cause breast cancer from the radiation.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: High**

Mammograms use a very low dose of radiation — far less than a single CT scan. The risk of radiation-induced cancer from mammography is extremely small. The benefits of early detection far outweigh this risk.

**Source:**...

Claim: Women cannot get pregnant while breastfeeding — it is 100% safe.
Response: **Verdict: False**
**Confidence: High** | **Evidence: A** | **Severity: Medium**

Exclusive breastfeeding provides some protection against ovulation, but it is not foolproof. Many women resume periods within 6 months and can conceive while breastfeeding. The Lactational Amenorrhoea Method (LAM) is 9...


eval/loss,1.02683
eval/runtime,14.2932
eval/samples_per_second,4.198
eval/steps_per_second,2.099
total_flos,5494842306674688
train/epoch,2.66667
train/global_step,80
train/grad_norm,1.37586
train/learning_rate,0.0
train/loss,0.8919
+4,...


Run 2 done.
